In [ ]:
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from openai import OpenAI
from google.colab import userdata

# ── MODEL SELECTION ───────────────────────────────────────────────────────────
# Change this between runs — must match the Task 1 model you're labeling.
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_NAME = "DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_SHORT = MODEL_NAME.split("/")[-1]

# ── GPT-4o CONFIG ─────────────────────────────────────────────────────────────
OPENAI_API_KEY      = userdata.get('OPENAI_API_KEY')
GPT_MODEL           = "gpt-4o"
CONFIDENCE_THRESHOLD = 0.8    # minimum GPT-4o confidence to keep a candidate
CONTEXT_WINDOW       = 80     # tokens of context each side for GPT-4o prompt
                               # (in words, not model tokens — approximate)

# ── LABELING CONFIG ───────────────────────────────────────────────────────────
WINDOW_TOKENS        = 30     # ±30 tokens around revision for activation extraction
MIN_REVISIONS        = 1      # drop traces with fewer than this many verified revisions
MAX_CANDIDATES_PER_TRACE = 50 # cap Stage 1 candidates to control GPT-4o cost

# ── PATHS ─────────────────────────────────────────────────────────────────────
GDRIVE_BASE   = Path("/content/gdrive/MyDrive/llm_planning_study")
TRACE_DIR     = GDRIVE_BASE / "traces" / MODEL_SHORT
LABEL_DIR     = GDRIVE_BASE / "labels" / MODEL_SHORT
LOG_FILE      = LABEL_DIR / "labeling_log.json"
SUMMARY_FILE  = GDRIVE_BASE / "labels_summary_all_models.json"

print(f"Model:       {MODEL_SHORT}")
print(f"Trace dir:   {TRACE_DIR}")
print(f"Label dir:   {LABEL_DIR}")


# -----------------------------------------------------------------------------
# CELL 3 — Mount Google Drive
# -----------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")
LABEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Label directory ready: {LABEL_DIR}")


# -----------------------------------------------------------------------------
# CELL 4 — OpenAI client
# -----------------------------------------------------------------------------
client = OpenAI(api_key=OPENAI_API_KEY)

def test_openai_connection():
    try:
        r = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": "Reply with: OK"}],
            max_tokens=5
        )
        print(f"OpenAI connection OK: {r.choices[0].message.content.strip()}")
        return True
    except Exception as e:
        print(f"OpenAI connection FAILED: {e}")
        return False

assert test_openai_connection(), "Fix OpenAI API key before continuing."


# -----------------------------------------------------------------------------
# CELL 5 — Text cleaning
# -----------------------------------------------------------------------------
def clean_text(text: str) -> str:
    """Remove HuggingFace tokenizer artifacts."""
    text = text.replace("Ġ", " ")
    text = text.replace("Ċ", "\n")
    text = text.replace("ĉ", "\t")
    return text


def get_analysis_text(trace: dict) -> str:
    """
    Return the best text to analyze for revisions.
    Prefer think_content (R1 reasoning block) over response_clean.
    Falls back to response_clean for non-reasoning models.
    """
    think = clean_text(trace.get("think_content", ""))
    clean = clean_text(trace.get("response_clean", ""))
    return think if think.strip() else clean


# -----------------------------------------------------------------------------
# CELL 6 — Stage 1: Rule-based candidate detection
# -----------------------------------------------------------------------------

# Revision marker patterns — ordered roughly by specificity
# Each tuple: (pattern_string, revision_type_hint)
REVISION_PATTERNS = [
    # Strong global revision signals
    (r"let me (?:start|try|approach|think about) (?:this )?(?:again|differently|from scratch|over)",
     "global"),
    (r"(?:i need to|let me) (?:completely |totally )?reconsider",           "global"),
    (r"(?:actually|wait)[,.]?\s+(?:i think )?(?:my approach|this approach|my method) (?:is wrong|doesn'?t work|is incorrect)",
     "global"),
    (r"(?:let me|i'?ll) (?:try a |use a )?(?:different|new|another) (?:approach|method|strategy|way)",
     "global"),
    (r"i (?:made|think i made) (?:a )?(?:mistake|an error|a wrong assumption)",
     "global"),
    (r"this (?:doesn'?t|won'?t) work",                                      "global"),
    (r"let me (?:restart|start over|begin again)",                           "global"),
    (r"on second thought",                                                   "global"),

    # Moderate — could be local or global
    (r"wait,?\s+(?:actually|let me|no)",                                     "local"),
    (r"actually,?\s+(?:wait|no|let me|that'?s)",                             "local"),
    (r"(?:hmm+|hm)[,.]?\s+(?:actually|wait|let me|so)",                     "local"),
    (r"let me (?:re-?check|re-?examine|re-?verify|double.?check)",           "local"),
    (r"that'?s (?:not right|wrong|incorrect|not correct)",                   "local"),
    (r"i was wrong",                                                         "local"),
    (r"let me recalculate",                                                  "local"),
    (r"(?:wait|actually)[,.]?\s+i (?:think i )?(?:made an error|got that wrong|miscalculated)",
     "local"),

    # Weaker signals — require GPT-4o verification to confirm
    (r"\bwait\b(?!\s+for)",                                                  "local"),
    (r"\bactually\b",                                                        "local"),
    (r"let me (?:re-?think|think (?:about )?this (?:more carefully|again))", "local"),
    (r"alternatively[,.]",                                                   "local"),
    (r"no[,.]?\s+wait",                                                      "local"),
]

# Compile all patterns
COMPILED_PATTERNS = [
    (re.compile(pat, re.IGNORECASE), hint)
    for pat, hint in REVISION_PATTERNS
]


def stage1_candidates(text: str, max_candidates: int) -> list[dict]:
    """
    Stage 1: scan text for revision marker matches.
    Returns list of candidate dicts with char position and type hint.
    Deduplicates overlapping matches (keeps the most specific).
    """
    print(f"    [DEBUG] Has Ġ: {'Ġ' in text} | First 100: {repr(text[:100])}")

    candidates = []
    seen_positions = set()

    for pattern, type_hint in COMPILED_PATTERNS:
        for match in pattern.finditer(text):
            start = match.start()
            # Deduplicate: skip if a match already found within 20 chars
            if any(abs(start - s) < 20 for s in seen_positions):
                continue
            seen_positions.add(start)
            candidates.append({
                "char_start":    start,
                "char_end":      match.end(),
                "matched_text":  match.group(),
                "type_hint":     type_hint,
            })

    # Sort by position
    candidates.sort(key=lambda x: x["char_start"])

    # Cap to MAX_CANDIDATES_PER_TRACE to control GPT-4o cost
    if len(candidates) > max_candidates:
        # Keep a mix: first 20, last 10, random sample from middle
        import random
        mid = candidates[20:-10]
        random.shuffle(mid)
        candidates = candidates[:20] + mid[:max_candidates-30] + candidates[-10:]
        candidates.sort(key=lambda x: x["char_start"])

    return candidates[:max_candidates]


# -----------------------------------------------------------------------------
# CELL 7 — Stage 2: GPT-4o verification
# -----------------------------------------------------------------------------

VERIFY_SYSTEM_PROMPT = """You are analyzing excerpts from a language model's internal reasoning trace.
Your job is to determine whether the marked position represents a genuine plan revision.

A GENUINE REVISION is when the model:
- Realizes its current approach or calculation is wrong and decides to change strategy
- Abandons a line of reasoning and starts a genuinely different approach
- Corrects a fundamental assumption that changes the solution path

NOT a genuine revision:
- Minor rephrasing or clarification without changing approach
- Saying "actually" or "wait" as a filler before continuing the same reasoning
- A brief self-correction of a small arithmetic slip while keeping the same method
- Transition phrases that don't indicate a real change in approach

Classify each as LOCAL or GLOBAL revision, or NOT_A_REVISION:
- LOCAL: the model corrects a single step or calculation but keeps its overall strategy
- GLOBAL: the model abandons its entire approach and starts a fundamentally different strategy
- NOT_A_REVISION: no genuine revision is happening

Respond ONLY with a JSON object in this exact format:
{"verdict": "LOCAL" | "GLOBAL" | "NOT_A_REVISION", "confidence": 0.0-1.0, "reason": "one sentence"}"""


def get_context_window(text: str, char_pos: int, window_words: int = 80) -> tuple[str, str]:
    """
    Extract context around char_pos.
    Returns (before_context, after_context) as strings.
    Uses word-based windowing for natural sentence boundaries.
    """
    # Convert char position to approximate word position
    words_before = text[:char_pos].split()
    words_after  = text[char_pos:].split()

    before = " ".join(words_before[-window_words:])
    after  = " ".join(words_after[:window_words])
    return before, after


def verify_candidate(
    text: str,
    candidate: dict,
    problem_difficulty: str,
    retries: int = 3
) -> dict:
    """
    Stage 2: Ask GPT-4o whether a candidate is a genuine revision.
    Returns the candidate dict augmented with verdict, confidence, reason.
    Retries on API errors with exponential backoff.
    """
    before, after = get_context_window(text, candidate["char_start"])

    user_prompt = (
        f"Problem difficulty: {problem_difficulty}\n\n"
        f"--- CONTEXT BEFORE MARKED POSITION ---\n{before}\n\n"
        f"*** MARKED: '{candidate['matched_text']}' ***\n\n"
        f"--- CONTEXT AFTER MARKED POSITION ---\n{after}\n\n"
        f"Is this a genuine plan revision? Respond with the JSON object."
    )

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": VERIFY_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt}
                ],
                max_tokens=120,
                temperature=0.0,    # deterministic verification
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)

            # Validate response structure
            if "verdict" not in result or "confidence" not in result:
                raise ValueError(f"Unexpected response format: {result}")

            return {
                **candidate,
                "verdict":    result["verdict"],
                "confidence": float(result["confidence"]),
                "reason":     result.get("reason", ""),
                "verified":   True,
            }

        except Exception as e:
            wait = 2 ** attempt
            print(f"    GPT-4o error (attempt {attempt+1}/{retries}): {e}. "
                  f"Waiting {wait}s...")
            time.sleep(wait)

    # All retries failed — mark as unverified
    return {
        **candidate,
        "verdict":    "VERIFICATION_FAILED",
        "confidence": 0.0,
        "reason":     "API error after retries",
        "verified":   False,
    }


# -----------------------------------------------------------------------------
# CELL 8 — Stage 3: Token alignment
# -----------------------------------------------------------------------------

def char_to_word_index(text: str, char_pos: int) -> int:
    """
    Convert character position to approximate word index.
    Used to find the token immediately before the revision marker.
    """
    return len(text[:char_pos].split())


def find_control_token(
    text: str,
    revision_positions: list[int],
    revision_char_pos: int,
    min_distance_words: int = 50
) -> int:
    """
    Find a matched control token position for a revision token.
    Control must be:
    - Same trace
    - At least min_distance_words away from any revision
    - Not at the very start or end of the trace (first/last 10% excluded)
    """
    import random
    words = text.split()
    n     = len(words)
    excluded_start = int(n * 0.1)
    excluded_end   = int(n * 0.9)

    # Convert all revision positions to word indices
    rev_word_indices = [char_to_word_index(text, p) for p in revision_positions]

    # Find candidate control positions
    candidates = []
    for i in range(excluded_start, excluded_end, 10):  # step=10 for efficiency
        too_close = any(abs(i - r) < min_distance_words for r in rev_word_indices)
        if not too_close:
            candidates.append(i)

    if not candidates:
        # Fallback: just use midpoint of trace
        return n // 2

    return random.choice(candidates)


def align_tokens(
    text: str,
    candidate: dict,
    all_revision_char_positions: list[int]
) -> dict:
    """
    Stage 3: Compute exact word-level token indices for extraction.

    We store word indices rather than character indices because:
    - Tokenizers split words differently across models
    - Word indices are model-agnostic and can be converted in Task 3
    - The token immediately BEFORE the marker is what we extract
    """
    char_pos   = candidate["char_start"]
    word_idx   = char_to_word_index(text, char_pos)

    # Token immediately before marker (where prediction error should be strongest)
    revision_word_idx = max(0, word_idx - 1)

    # Matched control token
    control_word_idx = find_control_token(
        text,
        all_revision_char_positions,
        char_pos
    )

    return {
        **candidate,
        "revision_word_idx":  revision_word_idx,
        "control_word_idx":   control_word_idx,
        "window_tokens":      WINDOW_TOKENS,
    }


# -----------------------------------------------------------------------------
# CELL 9 — Full labeling pipeline for one trace
# -----------------------------------------------------------------------------

def label_trace(trace: dict) -> dict | None:
    """
    Run the full 3-stage pipeline on one trace.
    Returns a label dict, or None if the trace should be skipped.

    Skips:
    - Truncated traces (incomplete reasoning)
    - Traces with fewer than MIN_REVISIONS verified revisions
    - Llama-style non-reasoning traces with 0 candidates
    """
    pid        = trace["problem_id"]
    difficulty = trace.get("difficulty", "unknown")
    truncated  = trace.get("truncated", False)

    # Skip truncated traces
    if truncated:
        return {"problem_id": pid, "skipped": True, "reason": "truncated"}

    text = get_analysis_text(trace)
    if not text.strip():
        return {"problem_id": pid, "skipped": True, "reason": "empty_text"}

    # ── STAGE 1: Rule-based candidates ──────────────────────────────────────
    candidates = stage1_candidates(text, MAX_CANDIDATES_PER_TRACE)

    if not candidates:
        return {
            "problem_id": pid,
            "skipped":    True,
            "reason":     "no_candidates",
            "difficulty": difficulty,
        }

    print(f"    Stage 1: {len(candidates)} candidates found")

    # ── STAGE 2: GPT-4o verification ────────────────────────────────────────
    verified = []
    for i, candidate in enumerate(candidates):
        result = verify_candidate(text, candidate, difficulty)

        is_revision   = result["verdict"] in ("LOCAL", "GLOBAL")
        above_thresh  = result["confidence"] >= CONFIDENCE_THRESHOLD

        if is_revision and above_thresh:
            verified.append(result)
            marker = "✓"
        else:
            marker = "✗"

        print(f"    [{i+1}/{len(candidates)}] {marker} "
              f"{result['verdict']:20s} conf={result['confidence']:.2f} "
              f"'{result['matched_text'][:30]}'")

    print(f"    Stage 2: {len(verified)}/{len(candidates)} verified revisions")

    # Drop traces below minimum revision threshold
    if len(verified) < MIN_REVISIONS:
        return {
            "problem_id": pid,
            "skipped":    True,
            "reason":     f"insufficient_revisions ({len(verified)} < {MIN_REVISIONS})",
            "difficulty": difficulty,
            "n_candidates": len(candidates),
            "n_verified":   len(verified),
        }

    # ── STAGE 3: Token alignment ─────────────────────────────────────────────
    all_char_positions = [c["char_start"] for c in verified]
    aligned = []
    for rev in verified:
        aligned.append(align_tokens(text, rev, all_char_positions))

    # Count local vs global
    local_count  = sum(1 for r in aligned if r["verdict"] == "LOCAL")
    global_count = sum(1 for r in aligned if r["verdict"] == "GLOBAL")

    return {
        "problem_id":     pid,
        "model":          MODEL_SHORT,
        "difficulty":     difficulty,
        "level":          trace.get("level", "unknown"),
        "subject":        trace.get("subject", "unknown"),
        "skipped":        False,
        "n_candidates":   len(candidates),
        "n_verified":     len(verified),
        "n_local":        local_count,
        "n_global":       global_count,
        "response_tokens": trace.get("response_tokens", 0),
        "revisions":      aligned,   # list of verified, aligned revision dicts
        "timestamp":      datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# CELL 10 — Checkpoint log
# -----------------------------------------------------------------------------
def load_label_log() -> dict:
    if LOG_FILE.exists():
        with open(LOG_FILE) as f:
            log = json.load(f)
        print(f"Resuming. Labeled: {len(log['completed'])} | "
              f"Skipped: {len(log['skipped'])} | Failed: {len(log['failed'])}")
    else:
        log = {
            "model":     MODEL_SHORT,
            "started":   datetime.now().isoformat(),
            "completed": [],   # problem_ids successfully labeled
            "skipped":   [],   # problem_ids skipped with reason
            "failed":    [],   # problem_ids that errored
        }
        print("Starting fresh labeling run.")
    return log


def save_label_log(log: dict):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)


def save_label(label: dict):
    out = LABEL_DIR / f"{label['problem_id']}.json"
    with open(out, "w") as f:
        json.dump(label, f, indent=2, ensure_ascii=False)


# -----------------------------------------------------------------------------
# CELL 11 — Load traces and build work queue
# -----------------------------------------------------------------------------
def load_traces(trace_dir: Path) -> list[dict]:
    """Load all non-truncated, non-log trace files."""
    files = [f for f in trace_dir.glob("*.json") if f.name != "run_log.json"]
    traces = []
    for f in files:
        with open(f) as fp:
            t = json.load(fp)
        traces.append(t)
    print(f"Loaded {len(traces)} traces from {trace_dir}")
    return traces


log    = load_label_log()
traces = load_traces(TRACE_DIR)
traces.pop(300)

done_ids = set(log["completed"] + log["skipped"] + log["failed"])
remaining = [t for t in traces if t["problem_id"] not in done_ids]
print(f"Traces remaining to label: {len(remaining)}")

# Show breakdown by difficulty
diff_counts = defaultdict(int)
for t in remaining:
    diff_counts[t.get("difficulty", "unknown")] += 1
for diff, cnt in diff_counts.items():
    print(f"  {diff}: {cnt}")


# -----------------------------------------------------------------------------
# CELL 12 — Main labeling loop
# -----------------------------------------------------------------------------
print(f"\nStarting labeling: {len(remaining)} traces")
print(f"Model: {MODEL_SHORT}")
print(f"GPT-4o model: {GPT_MODEL}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print("=" * 65)

for i, trace in enumerate(remaining):
    pid  = trace["problem_id"]
    diff = trace.get("difficulty", "?")
    print(f"\n[{i+1}/{len(remaining)}] {pid}  diff={diff}  "
          f"tokens={trace.get('response_tokens', '?')}")

    try:
        label = label_trace(trace)

        if label is None:
            log["failed"].append(pid)
            save_label_log(log)
            continue

        save_label(label)

        if label.get("skipped"):
            log["skipped"].append(pid)
            print(f"  SKIPPED: {label.get('reason', 'unknown')}")
        else:
            log["completed"].append(pid)
            print(f"  ✓ {label['n_verified']} revisions "
                  f"({label['n_local']} local, {label['n_global']} global)")

        save_label_log(log)

    except Exception as e:
        print(f"  FAILED: {e}")
        log["failed"].append(pid)
        save_label_log(log)

print("\n" + "=" * 65)
print(f"Labeling complete.")
print(f"  Completed : {len(log['completed'])}")
print(f"  Skipped   : {len(log['skipped'])}")
print(f"  Failed    : {len(log['failed'])}")


# -----------------------------------------------------------------------------
# CELL 13 — Label summary analysis
# Run after labeling completes for this model.
# Produces the stats you need for the paper's experimental setup section.
# -----------------------------------------------------------------------------
def label_summary(label_dir: Path, model_short: str) -> dict:
    files = [f for f in label_dir.glob("*.json")
             if f.name != "labeling_log.json"]

    if not files:
        print("No label files found.")
        return {}

    by_diff = defaultdict(lambda: {
        "n_labeled": 0, "n_skipped": 0,
        "total_revisions": [], "local": [], "global_": [],
        "skip_reasons": defaultdict(int)
    })

    for f in files:
        with open(f) as fp:
            label = json.load(fp)

        diff = label.get("difficulty", "unknown")

        if label.get("skipped"):
            by_diff[diff]["n_skipped"] += 1
            by_diff[diff]["skip_reasons"][label.get("reason", "unknown")] += 1
        else:
            by_diff[diff]["n_labeled"] += 1
            by_diff[diff]["total_revisions"].append(label.get("n_verified", 0))
            by_diff[diff]["local"].append(label.get("n_local", 0))
            by_diff[diff]["global_"].append(label.get("n_global", 0))

    print(f"\n{'='*65}")
    print(f"LABEL SUMMARY — {model_short}")
    print(f"{'='*65}")

    summary = {"model": model_short, "by_difficulty": {}}

    for diff in ["hard", "medium"]:
        if diff not in by_diff:
            continue
        d  = by_diff[diff]
        nl = d["n_labeled"]
        ns = d["n_skipped"]

        if nl == 0:
            print(f"\n  {diff.upper()}: all {ns} traces skipped")
            continue

        avg_rev  = sum(d["total_revisions"]) / nl
        avg_loc  = sum(d["local"]) / nl
        avg_glob = sum(d["global_"]) / nl

        print(f"\n  {diff.upper()}")
        print(f"    Labeled traces:      {nl}")
        print(f"    Skipped traces:      {ns}")
        print(f"    Avg revisions/trace: {avg_rev:.1f}")
        print(f"    Avg local/trace:     {avg_loc:.1f}")
        print(f"    Avg global/trace:    {avg_glob:.1f}")
        print(f"    Skip reasons:")
        for reason, cnt in d["skip_reasons"].items():
            print(f"      {reason}: {cnt}")

        summary["by_difficulty"][diff] = {
            "n_labeled":    nl,
            "n_skipped":    ns,
            "avg_revisions": round(avg_rev, 1),
            "avg_local":    round(avg_loc, 1),
            "avg_global":   round(avg_glob, 1),
        }

    # Save to Drive
    out = LABEL_DIR / "label_summary.json"
    with open(out, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  Summary saved: {out}")

    # Append to cross-model summary
    if SUMMARY_FILE.exists():
        with open(SUMMARY_FILE) as f:
            cross = json.load(f)
    else:
        cross = {}
    cross[model_short] = summary
    with open(SUMMARY_FILE, "w") as f:
        json.dump(cross, f, indent=2)
    print(f"  Cross-model summary updated: {SUMMARY_FILE}")

    return summary


label_summary(LABEL_DIR, MODEL_SHORT)

Model:       DeepSeek-R1-Distill-Qwen-1.5B
Trace dir:   /content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Qwen-1.5B
Label dir:   /content/gdrive/MyDrive/llm_planning_study/labels/DeepSeek-R1-Distill-Qwen-1.5B
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Label directory ready: /content/gdrive/MyDrive/llm_planning_study/labels/DeepSeek-R1-Distill-Qwen-1.5B
OpenAI connection OK: OK
Starting fresh labeling run.
Loaded 301 traces from /content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Qwen-1.5B
Traces remaining to label: 300
  hard: 200
  medium: 100

Starting labeling: 300 traces
Model: DeepSeek-R1-Distill-Qwen-1.5B
GPT-4o model: gpt-4o
Confidence threshold: 0.8

[1/300] hard_04640  diff=hard  tokens=1867
    [DEBUG] Has Ġ: False | First 100: "Okay, so I have this polynomial: 9x⁴ + a₃x³ + a₂x² + a₁x + 15 = 0. It's a quartic polynomial with in"
    Stage 1: 1 candidat

{'model': 'DeepSeek-R1-Distill-Qwen-1.5B',
 'by_difficulty': {'hard': {'n_labeled': 84,
   'n_skipped': 21,
   'avg_revisions': 7.8,
   'avg_local': 2.4,
   'avg_global': 5.4},
  'medium': {'n_labeled': 44,
   'n_skipped': 41,
   'avg_revisions': 5.8,
   'avg_local': 1.4,
   'avg_global': 4.4}}}

In [ ]:
x = True
g = 1
for i in traces:
  if 'problem_id' not in i.keys():
    x = False
    print(g)
  g+=1
print(x)

In [ ]:
from pathlib import Path
import json

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")

# List of models to reset Task 2 checkpoints for
# Remove any you don't want to reset
models_to_reset = [
    "DeepSeek-R1-Distill-Llama-8B",
    "DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Qwen-14B",
    "DeepSeek-R1-Distill-Qwen-1.5B",
]

for model_short in models_to_reset:
    label_dir = GDRIVE_BASE / "labels" / model_short
    log_file  = label_dir / "labeling_log.json"

    if log_file.exists():
        log_file.unlink()
        print(f"  ✓ Deleted checkpoint: {model_short}")
    else:
        print(f"  - No checkpoint found: {model_short}")

    # Also delete all label files so Task 2 starts completely fresh
    label_files = [f for f in label_dir.glob("*.json")
                   if f.name not in ("label_summary.json",)]
    for f in label_files:
        f.unlink()
    if label_files:
        print(f"    Deleted {len(label_files)} label files")

print("\nDone. Task 2 will start fresh for all models.")

  ✓ Deleted checkpoint: DeepSeek-R1-Distill-Llama-8B
    Deleted 300 label files
  ✓ Deleted checkpoint: DeepSeek-R1-Distill-Qwen-7B
    Deleted 300 label files
  ✓ Deleted checkpoint: DeepSeek-R1-Distill-Qwen-14B
    Deleted 300 label files
  ✓ Deleted checkpoint: DeepSeek-R1-Distill-Qwen-1.5B
    Deleted 300 label files

Done. Task 2 will start fresh for all models.


In [ ]:
# Test on one no_candidates trace directly

# Populate no_cand list by reading existing label files
no_cand = []
for label_file in LABEL_DIR.glob("*.json"):
    try:
        with open(label_file) as f:
            label_data = json.load(f)
        if label_data.get("skipped") and label_data.get("reason") == "no_candidates":
            no_cand.append(label_data["problem_id"])
    except Exception as e:
        print(f"Error reading {label_file}: {e}")

if not no_cand:
    print("No 'no_candidates' traces found in label files. Cannot run test.")
else:
    pid = no_cand[0]
    with open(TRACE_DIR / f"{pid}.json") as f:
        trace = json.load(f)

    text = get_analysis_text(trace)
    print(f"After get_analysis_text:")
    print(f"  Has Ġ: {'Ġ' in text}")
    print(f"  First 200: {repr(text[:200])}")

    candidates = stage1_candidates(text, MAX_CANDIDATES_PER_TRACE)
    print(f"  Candidates found: {len(candidates)}")

In [ ]:
from google.colab import runtime
runtime.unassign()